## HW6 — мультиколлинеарность, масштабирование и регуляризация

Сравните линейную регрессию на подготовленных признаках и на таблице с 20 дополнительными колонками. Используйте 4-фолдную кросс-валидацию и MSE для `log_trip_duration` (MSLE исходного таргета). Основа — `Scaling/HW_6.ipynb`; полный CSV не пересчитывался.

In [ ]:
import numpy as np
import pandas as pd

processed_data = pd.read_csv('mydata.csv', index_col='id')

In [ ]:
processed_data.head()

### Порядок объектов

В исходном задании указано не перемешивать данные. Сохраните `KFold(n_splits=4)` без shuffle.

In [ ]:
from sklearn.model_selection import KFold

selector = KFold(n_splits=4)

### Шаг №5
### Замерьте качество (MSLE, как и раньше) на Кросс-валидации,
### используя MSE от log_trip_duration и назначенный selector
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Отделяем признаки от целевой переменной
X = processed_data.drop('log_trip_duration', axis=1)
y = processed_data['log_trip_duration']

# Создаём модель линейной регрессии
model = LinearRegression()

# Вычисляем ошибку на каждом фолде
scores = cross_val_score(
    model,
    X,
    y,
    cv=selector,
    scoring='neg_mean_squared_error'
)

# Получаем среднюю ошибку
cross_val_error = -scores.mean()

print(f"MSLE на Кросс-валидации: {cross_val_error:.3f}")


print(f"MSLE на Кросс-валидации: {round(cross_val_error, 3)}")

### Дополнительные признаки

В `new_data.csv` к исходным признакам добавлены 20 колонок. Проверьте на том же разбиении, улучшают ли они качество линейной регрессии.

In [ ]:
new_data = pd.read_csv('new_data.csv', index_col='id')

In [ ]:
new_data.head()

In [ ]:
### Шаг №6
### Замерьте качество (MSLE, как и раньше) на Кросс-валидации,
### используя MSE от log_trip_duration и назначенный ранее selector
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
### Your code is here
selector = KFold(n_splits=4)

model = LinearRegression()
X = new_data.drop('log_trip_duration', axis=1)
y = new_data['log_trip_duration']

scores = cross_val_score(
    model,
    X,
    y,
    cv=selector,
    scoring='neg_mean_squared_error'
)
cross_val_error_2 = -scores.mean()

print(f"MSLE на Кросс-валидации: {round(cross_val_error_2, 3)}")

### Ранг и число признаков

Для обеих матриц признаков посчитайте `numpy.linalg.matrix_rank` и количество столбцов, исключив таргет. В упражнении отдельную константу учитывать не требуется. Сопоставление этих чисел помогает заметить линейные зависимости; численный ранг также чувствителен к масштабу колонок.

In [ ]:
### Создайте переменные rank_processed, rank_new
### Соответственно равные рангу изначальной матрицы
### с данными и рангу матрицы третьего гнома

### Your code is here
X1 = processed_data.drop('log_trip_duration', axis=1)
rank_processed = np.linalg.matrix_rank(X1)

X2 = new_data.drop('log_trip_duration', axis=1)
rank_new = np.linalg.matrix_rank(X2)

In [ ]:
### Создайте переменные num_features_processed, num_features_new
### Соответственно равные кол-ву фичей в изначальной матрицы
### с данными и кол-ву фичей у третьего гнома

### Your code is here
num_features_processed = X1.shape[1]

num_features_new = X2.shape[1]

In [ ]:
### Шаг №7
print(f"В первой модели всего фичей: {num_features_processed}, - а ранг равен {rank_processed}")

print(f"Во второй модели всего фичей: {num_features_new}, - а ранг равен {rank_new}")

### Ridge и Lasso

Подберите alpha для каждого метода так, чтобы средняя MSLE по четырём фолдам была строго меньше 0.4. Перед регуляризацией используйте MinMaxScaler. В каждом фолде обучайте scaler только на train и затем применяйте его к train и validation. Пример цикла и собственные переборы ниже сохранены; достижение порога на полном Taxi здесь не подтверждено.

In [ ]:
### Пример, как это можно сделать в цикле
### То есть обучить Lasso, учитывая масштабирование
### Исключительно на трейне на каждой итерации



X = new_data.drop('log_trip_duration', axis=1)
Y = new_data['log_trip_duration']

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Lasso

scores = []

for train_index, test_index in selector.split(X):

    X_train, X_test = X.values[train_index], X.values[test_index]
    Y_train, Y_test = Y.values[train_index], Y.values[test_index]

    ### Фитим исключительно на трейне!
    scaler = MinMaxScaler()
    scaler.fit(X_train)

    ### Применяем обученный scaler на трейн и тест
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ### max_iter иногда требуется ставить побольше,
    ### особенно когда данных много и/или они сложные
    ### этот параметр регулирует верхнюю границу кол-ва
    ### итераций во время обучения
    ### подробнее - в документации

    ### По дефолту здесь параметр регуляризации alpha=1

    model_lasso = Lasso(max_iter=100000)
    model_lasso.fit(X_train_scaled, Y_train)

    predictions = model_lasso.predict(X_test_scaled)

    scores.append(np.mean((predictions - Y_test)**2))


print(f"MSLE на Кросс-валидации равен: {np.mean(scores)}")

### P.S. если вы уже умеете работать с конструкциями
### Pipeline, GridSearchCV, cross_validate
### Можете использовать их. Мы познакомимся с ними позже.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge

X = new_data.drop('log_trip_duration', axis=1)
Y = new_data['log_trip_duration']

# Значения регуляризации, которые проверим
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

best_alpha = None
best_error = float('inf')

for alpha in alphas:
    scores = []

    for train_index, test_index in selector.split(X):
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]

        Y_train = Y.iloc[train_index]
        Y_test = Y.iloc[test_index]

        # Масштабирование только по обучающим данным
        scaler = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Обучение Ridge
        model = Ridge(alpha=alpha)
        model.fit(X_train_scaled, Y_train)

        predictions = model.predict(X_test_scaled)

        mse = np.mean((predictions - Y_test) ** 2)
        scores.append(mse)

    # Средняя ошибка по 4 фолдам
    mean_error = np.mean(scores)

    print(f"alpha={alpha}: MSLE={mean_error:.5f}")

    if mean_error < best_error:
        best_error = mean_error
        best_alpha = alpha

print(f"\nЛучшая alpha: {best_alpha}")
print(f"MSLE на кросс-валидации: {best_error:.5f}")
print(f"Условие MSLE < 0.4: {best_error < 0.4}")

In [ ]:
alphas = np.linspace(1e-5, 1e-4, 10)

results = []

for alpha in alphas:
    scores = []

    for train_index, test_index in selector.split(X):
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        Y_train = Y.iloc[train_index]
        Y_test = Y.iloc[test_index]

        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = Lasso(alpha=alpha, max_iter=100000)
        model.fit(X_train_scaled, Y_train)

        predictions = model.predict(X_test_scaled)
        scores.append(np.mean((predictions - Y_test) ** 2))

    mean_error = np.mean(scores)
    results.append((alpha, mean_error))

    print(
        f"alpha={alpha:.5f} | "
        f"MSLE={mean_error:.6f} | "
        f"Подходит: {mean_error < 0.4}"
    )

valid = [(a, e) for a, e in results if e < 0.4]

if valid:
    best_alpha, best_error = min(valid, key=lambda x: x[1])
    print(f"\nЛучшая alpha: {best_alpha:.5f}")
    print(f"Ошибка: {best_error:.6f}")
else:
    print("\nВ проверенном диапазоне условие не выполнено.")